# PixArt-Sigma Training Matrix Analysis

This notebook analyzes the `run_metadata.json` files generated by the 9-run training matrix (3 Ranks x 3 Data Scales). It plots the training loss curves and compares hardware resource usage.

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import IPython.display as display

ROOT = Path.cwd().parent.parent
MATRIX_DIR = ROOT / 'outputs' / 'matrix'

results = []
loss_histories = {}

for p in MATRIX_DIR.glob('*/run_metadata.json'):
    with open(p, 'r') as f:
        data = json.load(f)
        name = f"R{data['rank']}_N{data['num_images']}"
        results.append({
            'Rank': data['rank'],
            'Data Scale': data['num_images'],
            'Train Seconds': data['train_seconds'],
            'Peak VRAM (GB)': data['peak_allocated_vram_gb'],
            'Name': name
        })
        loss_histories[name] = data['loss_history']

df = pd.DataFrame(results).sort_values(by=['Data Scale', 'Rank'])
print(f"Loaded {len(df)} completed runs.")
display.display(df)

## Training Loss Comparison

In [ ]:
if len(loss_histories) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
    data_scales = [50, 100, 260]

    for i, scale in enumerate(data_scales):
        ax = axes[i]
        for rank in [4, 8, 16]:
            name = f"R{rank}_N{scale}"
            if name in loss_histories:
                ax.plot(loss_histories[name], label=f"Rank {rank}", alpha=0.8)
        
        ax.set_title(f"Loss Curves (Data Scale = {scale})")
        ax.set_xlabel("Optimizer Steps")
        if i == 0:
            ax.set_ylabel("Loss")
        ax.legend()
        ax.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.show()
else:
    print("No data available to plot.")

## Resource Usage Comparison

In [ ]:
if not df.empty:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    sns.barplot(data=df, x='Data Scale', y='Train Seconds', hue='Rank', ax=ax1, palette='viridis')
    ax1.set_title('Training Time by Rank and Data Scale')
    ax1.set_ylabel('Time (seconds)')

    sns.barplot(data=df, x='Data Scale', y='Peak VRAM (GB)', hue='Rank', ax=ax2, palette='magma')
    ax2.set_title('Peak VRAM by Rank and Data Scale')
    ax2.set_ylabel('VRAM (GB)')
    ax2.set_ylim(0, max(df['Peak VRAM (GB)']) * 1.2)

    plt.tight_layout()
    plt.show()
else:
    print("No data available to plot.")